# AIIJC 2026 · AIC — финальное решение

`PVT-v2-B2-li` (timm-энкодер с линейным вниманием) читает RGB-кадр; параллельно
нативные JPEG DCT-коэффициенты проходят через forensic-ветку (`ForensicFusion`)
и сливаются с признаками энкодера на страйдах 8/16/32. Дальше признаки проходят
через `ForensicDisentangle` (DG-Force: PFD/EFD бутылочные горлышки на страйдах
4/8/16/32 + cross-attention 16→32 на страйде 32, канальный гейт с нулевой
инициализацией) и декодируются `EMCADDecoder` в маску, плюс отдельная
классификационная голова (`GateHead`) — предсказывает, изменено ли изображение
вообще. Подробнее: `configs/experiments/disentangle.md`.

- `disentangle_b2_li760_r8_long` — базовое обучение (18 эпох) даёт модель, которая
  никогда не видела development/holdout; это осознанный выбор ради независимой
  выборки для подбора чекпоинта и порогов.
- `..._all_data_ft` дообучает 3 полных прохода на объединении train+development+
  holdout (после того как чекпоинт и пороги уже зафиксированы на предыдущей
  стадии).Файнтюн делается на всех данных, чтобы улучшить генерализацию модели.
- `..._all_data_hard_pixel_ft` добавляет к этому ещё 3 прохода с hard-pixel
  лоссом, ужесточающим градиент на границе маски.


In [ ]:
import numpy as np
import os
import shutil
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (ROOT / "src").is_dir():
    raise RuntimeError("Откройте ноутбук из корня репозитория curly-guacamole.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

env_path = ROOT / ".env"
if not env_path.exists() and (ROOT / ".env.example").exists():
    shutil.copy(ROOT / ".env.example", env_path)
    print("Создан .env из .env.example — при необходимости задайте GPU/batch/workers.")

import global_config

data_path = global_config.PathsConfig.current_data_path() if False else (ROOT / global_config.DATA_PATH)
protocol_path = ROOT / "runs" / "validation_protocol_20260908" / "protocol"
jpeg_pretrained = ROOT / "DCT_djpeg.pth"

print("Корень проекта:", ROOT)
print("Данные (AIIJC_DATA_PATH):", data_path, "— найдены:", (data_path / "train_stage1").exists())
print("Протокол валидации:", protocol_path, "— найден:", protocol_path.exists())
print("DCT_djpeg.pth (нужен только для стадии 1):", jpeg_pretrained, "— найден:", jpeg_pretrained.exists())


## Цепочка конфигов и стадий обучения

Финальный конфиг наследует (`extends`) цепочку из шести родителей — каждое звено
меняет ровно один аспект относительно родителя (архитектурная линия, не гиперпараметр):

```text
configs/baseline.yaml                              # PVT-v2-B2 + native JPEG + LocalFusion + EMCAD, 640, 6 эпох
  → configs/experiments/disentangle_fuse.yaml            # + DG-Force (PFD/EFD) на strides 4/8/16/32, режим fuse
    → configs/experiments/disentangle_fuse_cross32.yaml  # + cross-attention 16→32 на страйде 32
      → configs/experiments/disentangle_b2_li760_long.yaml     # энкодер pvt_v2_b2_li, RGB 760, 18 эпох (3 полных прохода)
        → configs/experiments/disentangle_b2_li760_r8_long.yaml   # disentangle_reduction=8 (бюджет: 99.705 GFLOPs при 760)
          → configs/experiments/disentangle_b2_li760_r8_all_data_ft.yaml         # дотюн 3×3 прохода на train+dev+holdout
            → configs/experiments/disentangle_b2_li760_r8_all_data_hard_pixel_ft.yaml  # + hard-pixel loss (ФИНАЛ)
```

Обучение состоит из **трёх реально запускаемых стадий** (промежуточные конфиги
выше — только наследование архитектуры, отдельно не тренируются):

| Стадия | Конфиг | Эпох | Старт | LR (enc/jpeg/head) | Что нового |
|---|---|---|---|---|---|
| A | `disentangle_b2_li760_r8_long` | 18 (15 sampled + 3 full-pass) | pretrained (RGB timm + `DCT_djpeg.pth`) | 1e-4 / 3e-4 / 3e-4 (baseline) | базовая архитектура на train, валидация на development |
| B | `disentangle_b2_li760_r8_all_data_ft` | 3 (все full-pass) | EMA чекпоинта A (`best.pt`) | 1e-5 / 3e-5 / 3e-5 | `train_all_data=true` — train+development+holdout |
| C (финал) | `disentangle_b2_li760_r8_all_data_hard_pixel_ft` | 3 (все full-pass) | EMA чекпоинта B, снятого после 3 эпох | те же 1e-5 / 3e-5 / 3e-5, `scheduler: none` | `loss.hard_pixel_weight=0.1` (доля 0.1, радиус исключения границы 2px) + Triton backward JPEG-ветки, foreach-нормализация градиентов |

Валидация (подбор чекпоинта/порогов по development) есть только на стадии A.
Стадии B и C — «слепой» дотюн на объединённых данных без валидации: это
осознанный трейд-офф «использовать всю размеченную выборку перед сдачей» за счёт
отсутствия отдельного контроля переобучения на этих двух стадиях — фиксируем это
явно, а не скрываем.

Сиды: `train.seed = 42` (не переопределяется ни в одном звене цепочки) фиксируются
для `random`/`numpy`/`torch` внутри `ExperimentRunner` (`set_random_seed`, см.
`src/training/base.py`) на старте каждой стадии автоматически — отдельно
фиксировать их в ноутбуке не нужно. `torch.use_deterministic_algorithms` не
включён (`deterministic=False` по умолчанию) — это осознанный компромисс в
пользу скорости (`cudnn.benchmark=True`), а не забытый шаг.


In [ ]:
from src.config import load_experiment_config

FINAL_CONFIG_PATH = ROOT / "configs" / "experiments" / "disentangle_b2_li760_r8_all_data_hard_pixel_ft.yaml"
final_cfg = load_experiment_config(FINAL_CONFIG_PATH)

print("run_name:", final_cfg.run_name)
print("seed:", final_cfg.seed)
print("encoder / image_size:", final_cfg.model.encoder, final_cfg.dataset.image_size)
print("disentangle levels/mode/reduction/cross:", final_cfg.model.disentangle_levels,
      final_cfg.model.disentangle_mode, final_cfg.model.disentangle_reduction,
      final_cfg.model.disentangle_cross_strides)
print("epochs / full_pass_epochs:", final_cfg.train.epochs, final_cfg.train.full_pass_epochs)
print("LR enc/jpeg/head:", final_cfg.train.encoder_lr, final_cfg.train.jpeg_lr, final_cfg.train.head_lr)
print("hard_pixel weight/fraction/radius:", final_cfg.loss.hard_pixel_weight,
      final_cfg.loss.hard_pixel_fraction, final_cfg.loss.hard_pixel_radius)
print("devices / batch / accumulation / amp:", final_cfg.train.devices, final_cfg.train.batch_size,
      final_cfg.train.grad_accum_steps, final_cfg.train.amp)


## `disentangle_b2_li760_r8_long` (18 эпох, обучение с нуля)

15 эпох по 24 000 сэмплированных примеров (25% негативов, кроп-аугментации), затем 3 полных прохода по train на полных кадрах без кропа. DG-Force работает на strides 4/8/16/32, cross-attention только 16→32 (страйд 8 стоил бы дополнительных ~15 GFLOPs — сознательно исключён). Резюмируется чекпоинтом `runs/disentangle_b2_li760_r8_long/ckpt/best.pt` (выбран по development AIC с весом малых масок 1.6) и `ckpt/last.pt`.

Повторный запуск этой ячейки **продолжает** обучение (`resume: true`) с последней сохранённой эпохи


In [ ]:
from src.training.engine import run_experiment

stage_a_cfg = load_experiment_config(ROOT / "configs/experiments/disentangle_b2_li760_r8_long.yaml")
print("Стадия A:", stage_a_cfg.run_name, "| эпох:", stage_a_cfg.train.epochs,
      "| полных проходов:", stage_a_cfg.train.full_pass_epochs)

run_a = run_experiment(stage_a_cfg)
run_a.summary


## `disentangle_b2_li760_r8_all_data_ft` (дотюн на всех данных, 3 полных прохода)

Старт: EMA-веса `runs/disentangle_b2_li760_r8_long/ckpt/best.pt`. Новый optimizer/ scheduler/EMA (дотюн — не resume того же run). `train.train_all_data=true`: обучение идёт на train + development + holdout, включая проверенные originals — валидации на этой стадии нет, чекпоинт и пороги уже зафиксированы на стадии A. LR понижен на порядок (1e-5 / 3e-5 / 3e-5), cosine-затухание до 2% исходного LR за 3 полных прохода.


In [ ]:
stage_b_cfg = load_experiment_config(ROOT / "configs/experiments/disentangle_b2_li760_r8_all_data_ft.yaml")
print("Стадия B:", stage_b_cfg.run_name, "| finetune_from:", stage_b_cfg.train.finetune_from,
      "| train_all_data:", stage_b_cfg.train.train_all_data)

source_ckpt = stage_b_cfg.paths.runs_path / stage_b_cfg.train.finetune_from
if not source_ckpt.is_file():
    raise FileNotFoundError(f"Нет чекпоинта стадии A: {source_ckpt}. Сначала выполните ячейку стадии A.")

run_b = run_experiment(stage_b_cfg)
run_b.summary


### Снапшот стадии B перед стадией C

Финальный конфиг ссылается на снимок стадии B `disentangle_b2_li760_r8_all_data_ft_ep3` (а не на `disentangle_b2_li760_r8_all_data_ft` напрямую) — так делается **копия**
завершённого чекпоинта под отдельным именем, прежде чем продолжать обучение дальше. Это защищает от ситуации, когда стадия C (или любой другой альтернативный дотюн от той же точки) случайно `resume`-нется в каталог стадии B и испортит его как самостоятельную, воспроизводимую промежуточную точку. Копия делается один раз — после того как стадия B выше уже отработала все 3/3 эпохи.


In [ ]:
stage_b_dir = ROOT / "runs" / "disentangle_b2_li760_r8_all_data_ft"
stage_b_snapshot_dir = ROOT / "runs" / "disentangle_b2_li760_r8_all_data_ft_ep3"
final_ckpt_dir = ROOT / "runs" / final_cfg.run_name / "ckpt"

if (final_ckpt_dir / "last.pt").exists():
    print("У финального run уже есть свой ckpt/last.pt — снапшот стадии B больше не нужен.")
elif stage_b_snapshot_dir.exists():
    print("Снапшот уже существует:", stage_b_snapshot_dir)
else:
    if not (stage_b_dir / "ckpt" / "last.pt").exists():
        raise FileNotFoundError(f"Стадия B ещё не завершена: нет {stage_b_dir / 'ckpt' / 'last.pt'}")
    shutil.copytree(stage_b_dir, stage_b_snapshot_dir)
    print("Скопировано:", stage_b_dir, "->", stage_b_snapshot_dir)


## 7. disentangle_b2_li760_r8_all_data_hard_pixel_ft`

Старт: EMA `runs/disentangle_b2_li760_r8_all_data_ft_ep3/ckpt/last.pt`. Новый optimizer/EMA. Ещё 3 полных прохода по train+development+holdout, те же LR, но **постоянные** (`scheduler: none`, без warmup) — на этой стадии расписание LR сознательно не перенастраивалось повторно, чтобы изолировать эффект hard-pixel-лосса от эффекта нового LR-расписания.

Loss = исходный (BCE+Dice+aux+DG-Force patch/edge) `+ 0.1 * (hard_positive_BCE + hard_negative_BCE)`, где для каждого изображения отдельно берутся худшие 10% допустимых пикселей внутри GT и вне GT(полоса ±2px вокруг границы исключена из допустимой области). Пустая допустимая область даёт нулевой вклад; для чистых (негативных) изображений работает только hard-negative часть. 

Дополнительно включены две GPU-оптимизации, не меняющие архитектуру/loss/LR (`configs/experiments/gpu_training_optimizations.md`): тайловый Triton backward для градиентов JPEG-весов вместо full-resolution one-hot тензора, и foreach-нормализация накопленных градиентов по группам device/dtype. Обе влияют только на скорость и порядок редукции чисел (возможны float-отличия), не на семантику обучения.


In [ ]:
final_run = run_experiment(final_cfg)
final_run.summary


## 8. Инференс и сборка submission

Т.к. на стадиях B и C не было валидации пороги (`mask_threshold`, `cls_threshold`, `min_area`) берём из **стадии A** — последнего run, где реально была валидация и подбор операционной точки по development AIC (вес малых масок 1.6) — и передаём их явно. Веса при этом берём финального чекпоинта (`ckpt/last.pt`, EMA приоритетнее raw).


In [ ]:
import json

from src.inference.predict import ThresholdConfig
from src.inference.submission import create_submission

stage_a_summary = json.loads((ROOT / "runs" / stage_a_cfg.run_name / "summary.json").read_text(encoding="utf-8"))
best = stage_a_summary["best"]
thresholds = ThresholdConfig(
    mask_threshold=float(best["mask_threshold"]),
    cls_threshold=float(best["cls_threshold"]),
    min_area=float(best["min_area"]),
    area_cap=float(best.get("area_cap", 0.0)),
    n_bins=stage_a_cfg.eval.n_bins,.
)
print("Пороги (со стадии A):", thresholds)

run_dir = ROOT / "runs" / final_cfg.run_name
output_dir = ROOT / "submissions" / final_cfg.run_name

csv_path = create_submission(run_dir, output_dir, thresholds=thresholds, checkpoint_name="last.pt")
print("submission.csv:", csv_path)
print("Маски:", output_dir / "predictions")


In [ ]:
# Упаковка в ZIP
import shutil

archive_path = shutil.make_archive(str(output_dir), "zip", root_dir=output_dir)
print("Готовый архив для отправки:", archive_path)
